# AdaBoost — A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting (1997)

_Papers · Classical ML_

**Train weak classifiers one at a time, each focused on the examples the last ones got wrong, then take a weighted vote — turning rules-of-thumb into one strong classifier.**

---

This notebook is a practice scaffold for the **AdaBoost — A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting (1997)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — NumPy

In [ ]:
import numpy as np
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# ---------- weighted decision stump (depth-1 tree) ----------
def fit_stump(X, y, w):
    """y in {-1,+1}; w a weight per example. Return (feat, thresh, polarity, predictions)."""
    N, F = X.shape
    best = (None, None, None, np.inf, None)         # (feat, thr, pol, err, pred)
    for f in range(F):
        vals = np.unique(X[:, f])
        thr = (vals[:-1] + vals[1:]) / 2 if len(vals) > 1 else vals   # midpoints
        for t in thr:
            for pol in (1, -1):
                pred = np.where(pol * (X[:, f] - t) >= 0, 1, -1)
                err = np.sum(w[pred != y])          # WEIGHTED error (eq: eps_t)
                if err < best[3]:
                    best = (f, t, pol, err, pred)
    return best

# ---------- AdaBoost from scratch (Figure 2, Steps 1-5) ----------
def adaboost_fit(X, y, T):
    N = X.shape[0]
    w = np.full(N, 1.0 / N)                         # Step 1: equal weights
    stumps, alphas = [], []
    for _ in range(T):
        f, thr, pol, eps, pred = fit_stump(X, y, w) # Steps 1-3 (normalized via w already summing ~1)
        eps = min(max(eps, 1e-12), 1 - 1e-12)       # clip to avoid log(0)/div-by-0
        alpha = 0.5 * np.log((1 - eps) / eps)        # Step 4: alpha_t = 0.5 ln((1-eps)/eps)
        w = w * np.exp(-alpha * y * pred)            # Step 5: heavier on the misclassified
        w = w / w.sum()                              # renormalize to a distribution
        stumps.append((f, thr, pol)); alphas.append(alpha)
    return stumps, alphas

def predict(X, stumps, alphas, upto=None):
    agg = np.zeros(X.shape[0])
    for (f, thr, pol), a in zip(stumps[:upto], alphas[:upto]):
        agg += a * np.where(pol * (X[:, f] - thr) >= 0, 1, -1)
    return np.sign(agg)                              # weighted-majority vote h_f

# ---------- toy 2-D set: a vertical BAND (two edges -> one stump can't do it) ----------
rng = np.random.default_rng(3)
N = 300
X = rng.uniform(-3, 3, (N, 2))
y = np.where((X[:, 0] > -1.0) & (X[:, 0] < 1.4), 1, -1)   # +1 inside the band

# ---------- single stump vs the boosted ensemble ----------
S, A = adaboost_fit(X, y, 50)
acc1  = np.mean(predict(X, S, A, upto=1) == y)
acc50 = np.mean(predict(X, S, A) == y)
print("single stump  accuracy:", round(acc1, 4))    # ~0.7533  (one cut can't bracket a band)
print("ensemble (50) accuracy:", round(acc50, 4))   # 1.0      (ensemble beats the stump)

# ---------- THE ORACLE: match scikit-learn round by round ----------
sk = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                        n_estimators=12, algorithm="SAMME", random_state=0).fit(X, y)
mine_staged = [round(np.mean(predict(X, S, A, upto=t) == y), 4) for t in range(1, 13)]
sk_staged   = [round(s, 4) for s in sk.staged_score(X, y)]
print("mine staged  1..12:", mine_staged)
print("sklearn 1..12:", sk_staged)
print("MATCH:", mine_staged == sk_staged)           # expect True

# ---------- recompute the worked one-round example ----------
Xe = np.array([[1.], [2.], [3.], [4.], [5.]]); ye = np.array([1, 1, -1, 1, -1])
w0 = np.full(5, 0.2)
f, thr, pol, eps, pred = fit_stump(Xe, ye, w0)
alpha = 0.5 * np.log((1 - eps) / eps)
w1 = w0 * np.exp(-alpha * ye * pred); w1 = w1 / w1.sum()
print("eps:", round(eps, 4), " alpha:", round(alpha, 4),
      " new weights:", [round(v, 3) for v in w1])    # eps 0.2, alpha 0.6931, w=[.125,.125,.125,.5,.125]

## Visualize the data & results

_On a toy 2-D 'band' set, how does AdaBoost's training accuracy grow with the number of stumps — and does our from-scratch version match scikit-learn's AdaBoostClassifier at every round?_

In [ ]:
import numpy as np
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

def fit_stump(X, y, w):
    F = X.shape[1]; best = (None, None, None, np.inf, None)
    for f in range(F):
        vals = np.unique(X[:, f]); thr = (vals[:-1]+vals[1:])/2 if len(vals)>1 else vals
        for t in thr:
            for pol in (1, -1):
                pred = np.where(pol*(X[:, f]-t) >= 0, 1, -1)
                err = np.sum(w[pred != y])
                if err < best[3]: best = (f, t, pol, err, pred)
    return best

def adaboost_fit(X, y, T):
    w = np.full(X.shape[0], 1.0/X.shape[0]); S, A = [], []
    for _ in range(T):
        f, thr, pol, eps, pred = fit_stump(X, y, w); eps = min(max(eps,1e-12),1-1e-12)
        a = 0.5*np.log((1-eps)/eps); w = w*np.exp(-a*y*pred); w /= w.sum()
        S.append((f, thr, pol)); A.append(a)
    return S, A

def predict(X, S, A, upto=None):
    agg = np.zeros(X.shape[0])
    for (f, thr, pol), a in zip(S[:upto], A[:upto]):
        agg += a*np.where(pol*(X[:, f]-thr) >= 0, 1, -1)
    return np.sign(agg)

rng = np.random.default_rng(3); N = 300
X = rng.uniform(-3, 3, (N, 2)); y = np.where((X[:,0] > -1.0) & (X[:,0] < 1.4), 1, -1)

S, A = adaboost_fit(X, y, 12)
mine = [round(np.mean(predict(X, S, A, upto=t) == y), 4) for t in range(1, 13)]
sk = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                        n_estimators=12, algorithm="SAMME", random_state=0).fit(X, y)
skv = [round(s, 4) for s in sk.staged_score(X, y)]
print("ours   :", mine)   # [0.7533, 0.6633, 1.0, 1.0, ...]
print("sklearn:", skv)    # identical
print("identical:", mine == skv)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In the worked example the misclassified example ended with weight 0.5. Verify that and explain why the next stump must do something different.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Right examples: $0.2\times e^{-0.6931}=0.2\times0.5=0.1$; wrong example: $0.2\times e^{+0.6931}=0.2\times2=0.4$. — _$e^{\pm0.6931}=2$ or $1/2$ since $\alpha=\tfrac12\ln4$._
- Sum $Z=0.1\cdot4+0.4=0.8$; normalize the wrong one: $0.4/0.8=0.5$. — _Weights must sum to 1._
- Now half the total weight sits on that one example. — _The next stump's weighted error is dominated by it, so it is pushed to classify it correctly._

**Answer:** New weights $[0.125,0.125,0.125,0.5,0.125]$. AdaBoost concentrates weight on the hard example, so successive stumps specialize on what earlier ones missed &mdash; that is the whole mechanism.

</details>

**Problem 2.** Ablation (1 vs many rounds): on the band data, a single stump scores 0.7533 but the boosted ensemble reaches 1.0. Why can't one stump match the ensemble, and why do 3 rounds suffice?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- The positive class is a band: $+1$ when feature 0 lies between two cut points. — _Its boundary needs TWO thresholds._
- A stump makes only ONE cut, so it can carve off at most one edge of the band &mdash; the other side is always misclassified. — _That caps a single stump near 0.75 here._
- Round 1 cuts the left edge; the right-edge errors gain weight; round 2 attacks them; a third stump and the weighted vote stitch the two cuts into the band. — _The vote of several stumps can represent the two-sided region one stump cannot._

**Answer:** One stump is structurally limited to a single threshold, so it tops out at 0.7533 on a two-edged band. Boosting reweights toward the missed edge each round; by round 3 the weighted majority of three stumps reproduces the band exactly (accuracy 1.0). This is the ensemble beating the single weak learner &mdash; and it matches scikit-learn round for round.

</details>

**Problem 3.** A weak classifier comes back with weighted error $\epsilon_t=0.4$. Compute its vote weight $\alpha_t$, and compare it to a stump with $\epsilon_t=0.1$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\alpha_t=\tfrac12\ln\frac{1-0.4}{0.4}=\tfrac12\ln1.5=\tfrac12(0.4055)=0.2027$. — _Plug into the formula._
- For $\epsilon_t=0.1$: $\alpha_t=\tfrac12\ln\frac{0.9}{0.1}=\tfrac12\ln9=\tfrac12(2.1972)=1.0986$. — _Smaller error, larger trust._

**Answer:** The barely-useful stump ($\epsilon=0.4$) gets a quiet vote $\alpha\approx0.20$; the accurate one ($\epsilon=0.1$) gets a loud vote $\alpha\approx1.10$ &mdash; about 5x louder. AdaBoost trusts accurate weak classifiers far more, and ignores ones at $\epsilon=0.5$ entirely ($\alpha=0$).

</details>